# Bonus B6 solution, notebook 2: choosing a filter, and rewording the staffing question

The worked answers to the challenge of `b6b_pinecone_at_the_desk.ipynb`. The setup repeats what the tasks need
from the notebook (the files, their vectors, the Chroma collection, the Pinecone index, and the helpers `embed`,
`in_chroma`, `in_pinecone` and `answer`), so this file runs on its own. It needs the same two keys, and it deletes
its index at the end.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The keys come from the environment or .env, and are never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."
os.environ.setdefault("PINECONE_API_KEY", local.get("PINECONE_API_KEY") or "")
assert os.environ["PINECONE_API_KEY"], "Put PINECONE_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")
print("Pinecone key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes
Pinecone key found: yes


In [2]:
import time
import chromadb
from openai import OpenAI, omit
from pinecone import Pinecone, DenseVectorQuery

client = OpenAI(timeout=60)
pc = Pinecone()
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))
names = [p.name for p in files]
texts = [p.read_text() for p in files]
kinds = [name.split("_")[0] for name in names]

vectors = [item.embedding for item in client.embeddings.create(model=EMBEDDING_MODEL, input=texts).data]
print(len(vectors), "vectors")

36 vectors


The same vectors go to both stores, as in Part B.

In [3]:
chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection("shop_files", configuration={"hnsw": {"space": "cosine"}})
collection.upsert(ids=names, documents=texts, metadatas=[{"kind": kind} for kind in kinds], embeddings=vectors)

INDEX_NAME = "ai-agents-b6-shop"

if not pc.indexes.exists(INDEX_NAME):
    pc.indexes.create(
        name=INDEX_NAME,
        schema={"fields": {"embedding": {"type": "dense_vector", "dimension": len(vectors[0]), "metric": "cosine"}}},
        deployment={"deployment_type": "managed", "cloud": "aws", "region": "us-east-1"},
    )

index = pc.index(INDEX_NAME)

documents = []
for name, vector, text, kind in zip(names, vectors, texts, kinds):
    documents.append({"_id": name, "embedding": vector, "text": text, "kind": kind})
index.documents.upsert(namespace="shop", documents=documents)

def stored(namespace):
    spaces = index.describe_index_stats().namespaces
    return spaces[namespace].vector_count if namespace in spaces else 0


while stored("shop") < len(documents):
    time.sleep(1)

print("in Chroma:", collection.count(), "| in Pinecone:", stored("shop"))

in Chroma: 36 | in Pinecone: 36


In [4]:
def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding


def in_chroma(vector, k=3, where=None):
    got = collection.query(query_embeddings=[vector], n_results=k, where=where)
    rows = []
    for file_id, distance, text in zip(got["ids"][0], got["distances"][0], got["documents"][0]):
        rows.append({"id": file_id, "cosine": round(1 - distance, 3), "text": text})
    return rows


def in_pinecone(vector, k=3, filter=None):
    got = index.documents.search(
        namespace="shop",
        top_k=k,
        score_by=[DenseVectorQuery(field="embedding", values=vector)],
        include_fields=["text"],
        filter=filter,
    )
    return [{"id": match.id, "cosine": round(match.score, 3), "text": match.text} for match in got.matches]

RULES = ("Answer the question using only the documents below. "
         "If they do not contain the answer, say you do not know.")


def answer(question, rows):
    packed = "\n\n".join(f"[{row['id']}]\n{row['text']}" for row in rows)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RULES},
            {"role": "user", "content": f"Documents:\n\n{packed}\n\nQuestion: {question}"},
        ],
        max_completion_tokens=200,
        reasoning_effort=EFFORT or omit,
    )
    return response.choices[0].message.content

---

## Task 1: Choose the filter that keeps the file with the answer

The question is embedded once. Then the same search runs twice on Pinecone with `k=1`, and only the filter changes.

In [5]:
SHIPPED = "When was the replacement for the warped vinyl shipped?"
s = embed(SHIPPED)

for kind in ["ticket", "policy"]:
    rows = in_pinecone(s, k=1, filter={"kind": {"$eq": kind}})
    print(kind, "only | file used:", rows[0]["id"], "| cosine:", rows[0]["cosine"])
    print("  ", answer(SHIPPED, rows))

ticket only | file used: ticket_06_germany_warped_vinyl.md | cosine: 0.581


   The replacement was shipped on 2012-07-20.


policy only | file used: policy_damaged_media.md | cosine: 0.47


   I don’t know. The document says warped vinyl is replaced at no charge, but it doesn’t specify when the replacement is shipped.


With the tickets-only filter the nearest file is `ticket_06_germany_warped_vinyl.md` (0.581), and the reply gives the date: 2012-07-20. With the policies-only filter the nearest file is `policy_damaged_media.md` (0.47). It says that warped vinyl is replaced at no charge and holds no date, so the reply is "I don’t know".

Both searches ran without an error and both returned a file about warped vinyl. The date is in a ticket, so the tickets-only filter was the right one for this question. A filter is a decision about where the answer can be, and Pinecone carries it out as given.

---

## Task 2: Repair the miss by rewording the question

*When is support staffed?* put the representatives file first. We ask for the same fact in other words.

In [6]:
HOURS = "What hours is support open?"
h = embed(HOURS)

print(f"{'Chroma':41} Pinecone")
for a, b in zip(in_chroma(h), in_pinecone(h)):
    print(f"{a['cosine']:.3f}  {a['id']:34} {b['cosine']:.3f}  {b['id']}")

Chroma                                    Pinecone


0.612  policy_support_hours.md            0.612  policy_support_hours.md
0.464  policy_support_reps.md             0.464  policy_support_reps.md
0.404  policy_escalations.md              0.404  policy_escalations.md


In [7]:
rows = in_pinecone(h, k=1)
print("file used:", rows[0]["id"])
print(answer(HOURS, rows))

file used: policy_support_hours.md


Support is open Monday to Friday, 09:00 to 17:00 in the shop’s local time, except on public holidays.


For *What hours is support open?* both stores put `policy_support_hours.md` first (0.612), with `policy_support_reps.md` second (0.464), and the reply from the nearest file alone gives the hours: Monday to Friday, 09:00 to 17:00.

Nothing in either store changed between Part C and here. The question changed, so its vector changed, and both stores ranked the new vector the same way. The miss of Part C came from the wording of the question and of the files, which is all the embedding model was given. That is where it is repaired, in Chroma and in Pinecone alike.

---

## Delete the index

In [8]:
pc.indexes.delete(INDEX_NAME)

print("exists:", pc.indexes.exists(INDEX_NAME))

exists: False


`exists: False`.